# **Initializing**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import xgboost as xgb

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from xgboost import XGBRegressor

%matplotlib inline


In [ ]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    """Target encoder.
    
    Replaces categorical column(s) with the mean target value for
    each category.

    """
    
    def __init__(self, cols=None):
        """Target encoder
        
        Parameters
        ----------
        cols : list of str
            Columns to target encode.  Default is to target 
            encode all categorical columns in the DataFrame.
        """
        if isinstance(cols, str):
            self.cols = [cols]
        else:
            self.cols = cols
        
        
    def fit(self, X, y):
        """Fit target encoder to X and y
        
        Parameters
        ----------
        X : pandas DataFrame, shape [n_samples, n_columns]
            DataFrame containing columns to encode
        y : pandas Series, shape = [n_samples]
            Target values.
            
        Returns
        -------
        self : encoder
            Returns self.
        """
        
        # Encode all categorical cols by default
        if self.cols is None:
            self.cols = [col for col in X 
                         if str(X[col].dtype)=='object']

        # Check columns are in X
        for col in self.cols:
            if col not in X:
                raise ValueError('Column \''+col+'\' not in X')

        # Encode each element of each column
        self.maps = dict() #dict to store map for each column
        for col in self.cols:
            tmap = dict()
            uniques = X[col].unique()
            for unique in uniques:
                tmap[unique] = y[X[col]==unique].mean()
            self.maps[col] = tmap
            
        return self

        
    def transform(self, X, y=None):
        """Perform the target encoding transformation.
        
        Parameters
        ----------
        X : pandas DataFrame, shape [n_samples, n_columns]
            DataFrame containing columns to encode
            
        Returns
        -------
        pandas DataFrame
            Input DataFrame with transformed columns
        """
        Xo = X.copy()
        for col, tmap in self.maps.items():
            vals = np.full(X.shape[0], np.nan)
            for val, mean_target in tmap.items():
                vals[X[col]==val] = mean_target
            Xo[col] = vals
        return Xo
            
            
    def fit_transform(self, X, y=None):
        """Fit and transform the data via target encoding.
        
        Parameters
        ----------
        X : pandas DataFrame, shape [n_samples, n_columns]
            DataFrame containing columns to encode
        y : pandas Series, shape = [n_samples]
            Target values (required!).

        Returns
        -------
        pandas DataFrame
            Input DataFrame with transformed columns
        """
        return self.fit(X, y).transform(X, y)

In [ ]:
!pip install category_encoders

# **Loading Data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = '/content/drive/My Drive/ML Challenge'
train_x_path = base_path + '/train_x.csv'
train_y_path = base_path + '/train_y.csv'
test_x_path = base_path + '/test_x.csv'
test_y_path = base_path + '/test_y.csv'

In [ ]:
train_x = pd.read_csv(train_x_path)
train_y = pd.read_csv(train_y_path)
test_x = pd.read_csv(test_x_path)

# **Examining the Dataset**

In [ ]:
train_y.info()

In [ ]:
train_y.status_group.value_counts()

In [ ]:
train_x.installer.value_counts()

In [ ]:
train_x.info()

In [ ]:
train_x.describe()

# **Preparing the y Variable**

*Label Encoding the y variable*

In [ ]:
train_y['status_group'].replace({'functional': 2, 'functional needs repair':1, 'non functional': 0}, inplace=True)

In [ ]:
train_y.status_group.unique()

*Removing the 'id' Column*

In [ ]:
train_y = train_y.drop(['id'], axis=1)

# **Preparing the x Variable**

In [ ]:
def drop_columns(x_train, x_test):
  drop_cols = ['id', 'extraction_type_group', 'extraction_type_class', 'num_private', 'region', 'recorded_by', 'payment', 'quality_group', 'quantity_group', 'source_type', 'source_class', 'waterpoint_type_group']
  x_train = x_train.drop(drop_cols, axis=1)
  x_test = x_test.drop(drop_cols, axis=1)

  return x_train, x_test

***Function to Process Numeric Data***

In [ ]:
def process_numerical(x_train, x_test):
  x_train['amount_tsh'] = (x_train['amount_tsh']-x_train['amount_tsh'].min())/(x_train['amount_tsh'].max()-x_train['amount_tsh'].min())
  x_test['amount_tsh'] = (x_test['amount_tsh']-x_test['amount_tsh'].min())/(x_test['amount_tsh'].max()-x_test['amount_tsh'].min())

  x_train['date_recorded'] = pd.to_datetime(x_train['date_recorded']).dt.year
  x_test['date_recorded'] = pd.to_datetime(x_test['date_recorded']).dt.year

  x_train['public_meeting'] = x_train['public_meeting'].astype('bool')
  x_test['public_meeting'] = x_test['public_meeting'].astype('bool')

  x_train['permit'] = x_train['permit'].astype('bool')
  x_test['permit'] = x_test['permit'].astype('bool')

  x_train['location'] = x_train['longitude']*x_train['latitude']
  x_test['location'] = x_test['longitude']*x_test['latitude']

  return x_train, x_test

***Function to Target Encode Columns***

In [ ]:
def process_target(x_train, y_train, x_test):

  target_columns = ['wpt_name', 'subvillage', 'lga', 'ward', 'scheme_name']
  
  te = TargetEncoder()
  x_train[target_columns] = te.fit_transform(x_train[target_columns], y_train)
  x_test[target_columns] = te.transform(x_test[target_columns])

  return x_train, y_train, x_test

***Function to One-Hot Encode Columns***

In [ ]:
def process_onehot(x):
  onehot_columns = ['basin', 'extraction_type', 'management_group', 'payment_type', 'waterpoint_type', 'funder', 'quantity', 'source', 'scheme_management', 'water_quality', 'management', 'installer']
  
  threshold = x.funder.value_counts()
  threshold_values = threshold[:5].index
  x['funder'] = x.funder.where(x.funder.isin(threshold_values), 'Other') 

  threshold = x.scheme_management.value_counts()
  threshold_values = threshold[:5].index
  x['scheme_management'] = x.scheme_management.where(x.scheme_management.isin(threshold_values), 'Other') 

  threshold = x.management.value_counts()
  threshold_values = threshold[:6].index
  x['management'] = x.management.where(x.management.isin(threshold_values), 'Other') 

  threshold = x.installer.value_counts()
  threshold_values = threshold[:6].index
  x['installer'] = x.installer.where(x.installer.isin(threshold_values), 'Other') 

  for column in onehot_columns:
    one_hot = pd.get_dummies(x[column], prefix=column)
    x = pd.concat([x, one_hot], axis=1)
  
  x = x.drop(onehot_columns, axis = 1)
  return x

***Function to Fill Missing Values***

In [ ]:
from numpy import nan
from pandas import read_csv
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

def fill_missing(x_train, y_train, x_test):
  imputer = SimpleImputer(missing_values=nan, strategy='median')
  imputer.fit(x_train, y_train)
  x = pd.DataFrame(imputer.fit_transform(x_train))
  x.columns = x_train.columns
  x.index = x_train.index

  xt = pd.DataFrame(imputer.transform(x_test))
  xt.columns = x_test.columns
  xt.index = x_test.index

  return x, xt

***Processing the x Variable***

In [ ]:
train_x.amount_tsh

In [ ]:
x1, xt1 = drop_columns(train_x, test_x)

In [ ]:
x2, y_train, xt2 = process_target(x1, train_y, xt1)

In [ ]:
train_size, test_size = train_x.shape[0], test_x.shape[0]
x = pd.concat([x2, xt2])

In [ ]:
x = process_onehot(x)

In [ ]:
x3, xt3 = x.iloc[:train_size,:], x.iloc[train_size:,:]

In [ ]:
x4, xt4 = process_numerical(x3, xt3)

In [ ]:
x5, xt5 = fill_missing(x4, y_train, xt4)

In [ ]:
x5.info()

In [ ]:
duplicated_columns_list = []
list_of_all_columns = list(x5.columns)
for column in list_of_all_columns:
    if list_of_all_columns.count(column) > 1 and not column in duplicated_columns_list:
        duplicated_columns_list.append(column)
duplicated_columns_list

In [ ]:
x_, x_t, y_, y_t = train_test_split(x5, train_y, test_size=0.33, random_state=42)

# **Testing the Performance of Machine Learning Models**

***Training XGBoost Model***

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance, XGBClassifier
from sklearn.metrics import f1_score
plt.rcParams["figure.figsize"] = (20,10)

xg_reg = xgb.XGBClassifier(base_score=0.5, booster='dart', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=0.4, gamma=0.0,
             importance_type='gain', learning_rate=0.05, max_delta_step=0,
             max_depth=3, min_child_weight=7, missing=None, n_estimators=100,
             n_jobs=1,num_class=3, nthread=None, objective='multi:softmax',
             random_state=0, reg_alpha=0, reg_lambda=1, scale_pos_weight=1,
             seed=None, silent=None, subsample=1, verbosity=1)
xg_reg.fit(x5, y_train)

predictions_xg = xg_reg.predict(x_t)
f1_score(y_t, predictions_xg, average=None)

***Training Logistic Regression Model***

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

lr = LogisticRegression(multi_class='multinomial')
lr.fit(x_, y_)

predictions_lr = lr.predict(x_t)
f1_score(y_t, predictions_lr, average=None)

***Training Random Forest Classifier Model***

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

rfc = RandomForestClassifier(n_estimators=1000, max_depth=None, min_samples_split=6, min_samples_leaf=2, 
                               max_features='auto', min_impurity_decrease=0.1, min_impurity_split=None, 
                               bootstrap=True, warm_start=True)
rfc.fit(x_, y_)

predictions_rfc = rfc.predict(x_t)
f1_score(y_t, predictions_rfc, average=None)

***Training k Nearest Neighbours Model***

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_, y_)

predictions_knn = knn.predict(x_t)
f1_score(y_t, predictions_knn, average=None)

***Training Naive Bayes Model***

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import f1_score
from sklearn.preprocessing import MinMaxScaler

scaler_m = MinMaxScaler()
scaler_m.fit(x_)
x_pos = scaler_m.transform(x_)
xt_pos = scaler_m.transform(x_t)

nb = CategoricalNB()
nb.fit(x_pos, y_)

predictions_nb = nb.predict(xt_pos)
f1_score(y_t, predictions_nb, average=None)

***Training Neural Network Model***

In [ ]:
import tensorflow as tf

def plot_curve(epochs, hist, list_of_metrics):
  plt.figure()
  plt.xlabel("Epoch")
  plt.ylabel("Value")
  for m in list_of_metrics:
    x = hist[m]
    plt.plot(epochs[1:], x[1:], label=m)
  plt.legend()

def create_model(learning_rate):
  model = tf.keras.models.Sequential()
  model.add(tf.keras.layers.Dense(units=1024, activation='relu'))
  model.add(tf.keras.layers.Dense(units=256, activation='relu'))
  model.add(tf.keras.layers.Dense(units=64, activation='relu'))
  model.add(tf.keras.layers.Dropout(rate=0.5))
  model.add(tf.keras.layers.Dense(units=3, activation='softmax'))      
  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                loss="categorical_crossentropy",
                 metrics=["accuracy",
                       tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall()])
  return model    


def train_model(model, train_features, train_label, epochs,
                batch_size=None, validation_split=0.1):
  
  history = model.fit(x=train_features, y=train_label, batch_size=batch_size,
                      epochs=epochs, shuffle=True, 
                      validation_split=validation_split)
  epochs = history.epoch
  hist = pd.DataFrame(history.history)
  return epochs, hist   

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_s = StandardScaler()
scaler_s.fit(x_)
x_std = scaler_s.transform(x_)
xt_std = scaler_s.transform(x_t)

y_cat = pd.get_dummies(y_['status_group'])
yt_cat = pd.get_dummies(y_t['status_group'])

x_std = np.asarray(x_std).astype('float32')
model = create_model(learning_rate=0.01)
epochs, hist = train_model(model, x_std, y_cat, epochs=100, batch_size=5000, validation_split=0.1)
list_of_metrics_to_plot = ['accuracy']
plot_curve(epochs, hist, list_of_metrics_to_plot)
print("\n Evaluate the new model against the test set:")
model.evaluate(x=xt_std, y=yt_cat, batch_size=5000)

# **Plotting Feature Importance**

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance, XGBClassifier
plt.rcParams["figure.figsize"] = (20,10)

xg_reg = xgb.XGBClassifier(base_score=0.5, booster='dart', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=0.4, gamma=0.0,
             importance_type='gain', learning_rate=0.05, max_delta_step=0,
             max_depth=3, min_child_weight=7, missing=None, n_estimators=100,
             n_jobs=1,num_class=3, nthread=None, objective='multi:softmax',
             random_state=0, reg_alpha=0, reg_lambda=1, scale_pos_weight=1,
             seed=None, silent=None, subsample=1, verbosity=1)
xg_reg.fit(x_, y_)
cross_val_score(xg_reg, x4, y_train, cv=5, scoring='accuracy')

plot_importance(xg_reg)
plt.show()

# **Training the Final Model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

rfc = RandomForestClassifier(n_estimators=800, max_depth=None, min_samples_split=6, min_samples_leaf=2, 
                               max_features='auto', min_impurity_decrease=0.1, min_impurity_split=None, 
                               bootstrap=True, warm_start=True)
rfc.fit(x5, train_y)

predictions_rfc = rfc.predict(xt5)

# **Saving the Final Results**

In [ ]:
y_test = pd.DataFrame(predictions_rfc)

In [ ]:
y_test
y_test.replace({2:'functional', 1:'functional needs repair', 0:'non functional'}, inplace=True)

In [ ]:
sub = pd.DataFrame()
sub['id'] = test_x['id']
sub['status_group'] = y_test

In [ ]:
sub.to_csv(test_y_path,index=False)